# 2 多层感知机

## 2.1 理论计算题

### 1. 非线性激活函数的重要性

**问题描述：** 假设一个具有单隐藏层的多层感知机，输入为 x，隐藏层没有激活函数（即线性激活），表达为 h = W1x + b1，输出层为 o = W2h + b2。请通过代数推导证明，该网络等价于一个单层神经网络，并写出等价后的权重矩阵 W′ 和偏置向量 b′。

**代数推导：**

给定隐藏层输出：
$h = W_1x + b_1$

给定输出层输出：
$o = W_2h + b_2$

将 $h$ 的表达式代入 $o$ 的表达式中：
$o = W_2(W_1x + b_1) + b_2$

展开等式：
$o = W_2W_1x + W_2b_1 + b_2$

我们可以将 $W_2W_1$ 视为一个新的权重矩阵 $W'$，将 $W_2b_1 + b_2$ 视为一个新的偏置向量 $b'$。

令 $W' = W_2W_1$
令 $b' = W_2b_1 + b_2$

则原网络可以简化为：
$o = W'x + b'$

**结论：**

这证明了当隐藏层没有非线性激活函数时，一个单隐藏层的多层感知机等价于一个单层神经网络。等价后的权重矩阵 $W'$ 为 $W_2W_1$，偏置向量 $b'$ 为 $W_2b_1 + b_2$。非线性激活函数对于多层感知机学习复杂非线性模式至关重要，否则无论网络有多少层，它都只能表示线性变换。

### 2. 激活函数性质分析

**问题描述：** 写出 Sigmoid(x) 和 tanh(x) 的数学表达式，并推导它们的导数 Sigmoid′(x) 和 tanh′(x) 与其函数自身的关系。

**Sigmoid 函数：**

数学表达式：
$\text{Sigmoid}(x) = \frac{1}{1 + e^{-x}}$

导数推导：
$\text{Sigmoid}'(x) = \frac{d}{dx} \left( \frac{1}{1 + e^{-x}} \right)$ 
$= -1 \cdot (1 + e^{-x})^{-2} \cdot (-e^{-x})$ 
$= \frac{e^{-x}}{(1 + e^{-x})^2}$ 
$= \frac{1}{1 + e^{-x}} \cdot \frac{e^{-x}}{1 + e^{-x}}$ 
$= \frac{1}{1 + e^{-x}} \cdot \left( 1 - \frac{1}{1 + e^{-x}} \right)$ 
$= \text{Sigmoid}(x) \cdot (1 - \text{Sigmoid}(x))$

**结论：** Sigmoid 函数的导数可以表示为其函数值与 (1 - 函数值) 的乘积。

**tanh 函数：**

数学表达式：
$\text{tanh}(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$

导数推导：
$\text{tanh}'(x) = \frac{d}{dx} \left( \frac{e^x - e^{-x}}{e^x + e^{-x}} \right)$ 
$= \frac{(e^x + e^{-x})(e^x + e^{-x}) - (e^x - e^{-x})(e^x - e^{-x})}{(e^x + e^{-x})^2}$ 
$= \frac{(e^x + e^{-x})^2 - (e^x - e^{-x})^2}{(e^x + e^{-x})^2}$ 
$= 1 - \left( \frac{e^x - e^{-x}}{e^x + e^{-x}} \right)^2$ 
$= 1 - \text{tanh}^2(x)$

**结论：** tanh 函数的导数可以表示为 1 减去其函数值的平方。


## 2.2 编程题

不使用深度学习框架的高级 API（仅使用 Tensor 基础算子如 `torch.matmul` 等），纯 NumPy 或 PyTorch 从零实现一个多分类（使用 Fashion-MNIST 数据集）的单隐藏层 MLP。

**实现要求：**
1. 手动初始化隐藏层参数 `W1, b1` 和输出层参数 `W2, b2`（提示：使用正态分布随机初始化）。
2. 实现 ReLU 激活函数的前向传播：`max(0, x)`。
3. 实现带有 Softmax 的交叉熵损失函数。
4. 编写训练循环，通过小批量随机梯度下降（Mini-batch SGD）手动更新参数。

In [ ]:
import torch
import torchvision
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt

# 1. 数据加载与预处理
# Fashion-MNIST 数据集
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

batch_size = 256
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. MLP 模型定义与初始化
input_size = 28 * 28  # 784
hidden_size = 256
output_size = 10      # 10 classes for Fashion-MNIST

# 手动初始化参数 (使用正态分布随机初始化)
W1 = torch.randn(input_size, hidden_size) * 0.01
b1 = torch.zeros(hidden_size)
W2 = torch.randn(hidden_size, output_size) * 0.01
b2 = torch.zeros(output_size)

# 确保参数需要梯度
W1.requires_grad_(True)
b1.requires_grad_(True)
W2.requires_grad_(True)
b2.requires_grad_(True)

# 3. 实现 ReLU 激活函数的前向传播
def relu(x):
    return torch.max(torch.tensor(0.0), x)

# 4. 实现带有 Softmax 的交叉熵损失函数
def softmax_crossentropy_loss(outputs, targets):
    # Softmax
    exp_outputs = torch.exp(outputs - torch.max(outputs, dim=1, keepdim=True).values) # 减去最大值以提高数值稳定性
    softmax_probs = exp_outputs / torch.sum(exp_outputs, dim=1, keepdim=True)

    # Cross-entropy loss
    # targets 是类别索引，需要转换为one-hot编码或使用torch.gather
    # 这里直接使用torch.gather来获取对应类别的概率，然后取对数
    log_probs = torch.log(softmax_probs + 1e-9) # 加上一个小的epsilon防止log(0)
    loss = -torch.gather(log_probs, 1, targets.unsqueeze(1)).mean()
    return loss

# 5. 编写训练循环，通过小批量随机梯度下降（Mini-batch SGD）手动更新参数
def train_mlp(
    W1, b1, W2, b2, train_loader, test_loader, num_epochs=10, lr=0.1
):
    train_losses = []
    test_accuracies = []

    for epoch in range(num_epochs):
        # 训练阶段
        W1.grad = None
        b1.grad = None
        W2.grad = None
        b2.grad = None
        for i, (images, labels) in enumerate(train_loader):
            images = images.view(-1, input_size) # 展平图像

            # 前向传播
            hidden = relu(torch.matmul(images, W1) + b1)
            outputs = torch.matmul(hidden, W2) + b2

            # 计算损失
            loss = softmax_crossentropy_loss(outputs, labels)

            # 反向传播
            loss.backward()

            # 手动更新参数
            with torch.no_grad():
                W1 -= lr * W1.grad
                b1 -= lr * b1.grad
                W2 -= lr * W2.grad
                b2 -= lr * b2.grad

        train_losses.append(loss.item())

        # 评估阶段
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.view(-1, input_size)
                hidden = relu(torch.matmul(images, W1) + b1)
                outputs = torch.matmul(hidden, W2) + b2
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)

        print(
            f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Test Accuracy: {accuracy:.2f}%"
        )
    return train_losses, test_accuracies

# 运行训练
num_epochs = 10
lr = 0.1
train_losses, test_accuracies = train_mlp(
    W1, b1, W2, b2, train_loader, test_loader, num_epochs, lr
)

# 绘制损失和准确率曲线
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), test_accuracies, label='Test Accuracy', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Test Accuracy over Epochs')
plt.legend()

plt.tight_layout()
plt.show()


# 3 模型选择，权重衰减和丢弃法

## 3.1 理论计算题

### 1. 过拟合与欠拟合

**问题描述：** 简述训练误差（Training Error）与泛化误差（Generalization Error）的区别。当一个模型的训练误差极低，但泛化误差很高时，模型处于什么状态？应该如何通过控制模型复杂度来缓解这一现象？

**解答：**

*   **训练误差（Training Error）：** 模型在用于训练的数据集上计算得到的误差。它反映了模型对训练数据的拟合程度。
*   **泛化误差（Generalization Error）：** 模型在未见过的测试数据（或真实分布的数据）上计算得到的期望误差。它反映了模型对新数据的预测能力。

**状态：** 当一个模型的训练误差极低，但泛化误差很高时，模型处于**过拟合（Overfitting）**状态。这意味着模型过度学习了训练数据中的噪声和细节，导致其失去了泛化到新数据的能力。

**缓解方法（控制模型复杂度）：**
1.  **减少模型参数数量：** 减少神经网络的层数或每层的神经元数量。
2.  **正则化（Regularization）：** 如 L1 或 L2 正则化（权重衰减），通过在损失函数中添加惩罚项来限制权重的大小，防止模型过度依赖某些特征。
3.  **丢弃法（Dropout）：** 在训练过程中随机将一部分神经元的输出置为 0，强制网络学习更鲁棒的特征表示。
4.  **提前停止（Early Stopping）：** 在验证集误差开始上升时停止训练，防止模型在训练集上过度拟合。
5.  **增加训练数据：** 更多的数据可以帮助模型学习到更具代表性的特征，减少过拟合的风险。

### 2. K 折交叉验证

**问题描述：** 阐述 K 折交叉验证（K-fold Cross-Validation）的具体实施算法步骤。

**解答：**

K 折交叉验证是一种评估模型泛化性能的方法，特别适用于数据量较小的情况。其具体实施步骤如下：

1.  **数据划分：** 将原始训练数据集随机划分为 $K$ 个大小相等的互斥子集（称为“折”或“fold”）。
2.  **迭代训练与评估：** 进行 $K$ 次迭代。在第 $i$ 次迭代中（$i = 1, 2, ..., K$）：
    *   将第 $i$ 个子集作为**验证集**（Validation Set）。
    *   将剩余的 $K-1$ 个子集组合起来作为**训练集**（Training Set）。
    *   在训练集上训练模型。
    *   在验证集上评估模型，记录评估指标（如误差、准确率等），记为 $E_i$。
3.  **计算平均性能：** 将 $K$ 次迭代得到的评估指标求平均值，作为模型最终的性能评估结果：$E_{cv} = \frac{1}{K} \sum_{i=1}^{K} E_i$。

通过 K 折交叉验证，每个样本都有机会作为验证集被评估一次，从而更充分地利用了数据，得到了更稳定、可靠的模型性能估计。

## 3.2 编程题

在你实现的 MLP 上，加入 L2 正则化和 Dropout 机制。

**实现要求：**
1. 权重衰减：在你的自定义 SGD 优化器中，加入权重衰减。即在计算梯度更新时，让旧权重首先乘以 `(1 − ηλ)`。
2. Dropout 从零实现：编写一个 `dropout_layer(X, dropout)` 函数。根据传入的概率，利用随机掩码（Mask）将输入张量某些元素置 0，并进行缩放。注意：通过一个布尔变量（如 `is_training`）来控制测试时不触发 Dropout。
3. 对比实验：设计高维多项式拟合或使用极少样本训练一个复杂的 MLP，绘制并对比：1) 无正则化、2) 有权重衰减、3) 有 Dropout 三种情况下的训练和验证误差曲线（Loss Curve）。

In [ ]:
import torch
import torchvision
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt

# 1. 数据加载与预处理 (使用极少样本来模拟过拟合)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# 使用较小的训练集来容易产生过拟合
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
# 只取前 1000 个样本作为训练集
subset_indices = list(range(1000))
train_subset = torch.utils.data.Subset(train_dataset, subset_indices)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

batch_size = 64
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Dropout 从零实现
def dropout_layer(X, dropout, is_training=True):
    assert 0 <= dropout <= 1
    if dropout == 1:
        return torch.zeros_like(X)
    if dropout == 0 or not is_training:
        return X
    mask = (torch.rand(X.shape) > dropout).float()
    return mask * X / (1.0 - dropout)

# 3. 带有 L2 正则化和 Dropout 的 MLP 训练函数
def train_mlp_regularized(
    train_loader, test_loader, num_epochs=20, lr=0.1, weight_decay=0.0, dropout_rate=0.0
):
    input_size = 28 * 28
    hidden_size = 256
    output_size = 10

    # 初始化参数
    W1 = torch.randn(input_size, hidden_size) * 0.01
    b1 = torch.zeros(hidden_size)
    W2 = torch.randn(hidden_size, output_size) * 0.01
    b2 = torch.zeros(output_size)

    W1.requires_grad_(True)
    b1.requires_grad_(True)
    W2.requires_grad_(True)
    b2.requires_grad_(True)

    train_losses = []
    test_losses = []

    for epoch in range(num_epochs):
        # 训练阶段
        epoch_train_loss = 0.0
        num_batches = 0
        for images, labels in train_loader:
            images = images.view(-1, input_size)

            # 前向传播 (带 Dropout)
            hidden = relu(torch.matmul(images, W1) + b1)
            hidden = dropout_layer(hidden, dropout_rate, is_training=True)
            outputs = torch.matmul(hidden, W2) + b2

            # 计算损失
            loss = softmax_crossentropy_loss(outputs, labels)

            # 反向传播
            W1.grad = None
            b1.grad = None
            W2.grad = None
            b2.grad = None
            loss.backward()

            # 手动更新参数 (带权重衰减)
            with torch.no_grad():
                # 权重衰减：W = W - lr * grad - lr * lambda * W = W * (1 - lr * lambda) - lr * grad
                W1.copy_(W1 * (1 - lr * weight_decay) - lr * W1.grad)
                b1.copy_(b1 - lr * b1.grad) # 偏置通常不进行权重衰减
                W2.copy_(W2 * (1 - lr * weight_decay) - lr * W2.grad)
                b2.copy_(b2 - lr * b2.grad)

            epoch_train_loss += loss.item()
            num_batches += 1

        train_losses.append(epoch_train_loss / num_batches)

        # 评估阶段
        epoch_test_loss = 0.0
        num_test_batches = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.view(-1, input_size)
                # 测试时不使用 Dropout
                hidden = relu(torch.matmul(images, W1) + b1)
                hidden = dropout_layer(hidden, dropout_rate, is_training=False)
                outputs = torch.matmul(hidden, W2) + b2
                loss = softmax_crossentropy_loss(outputs, labels)
                epoch_test_loss += loss.item()
                num_test_batches += 1
        test_losses.append(epoch_test_loss / num_test_batches)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_losses[-1]:.4f}, Test Loss: {test_losses[-1]:.4f}")

    return train_losses, test_losses

# 4. 对比实验
num_epochs = 30
lr = 0.1

print("Training without Regularization...")
train_loss_base, test_loss_base = train_mlp_regularized(
    train_loader, test_loader, num_epochs=num_epochs, lr=lr, weight_decay=0.0, dropout_rate=0.0
)

print("\nTraining with Weight Decay (L2 Regularization)...")
train_loss_wd, test_loss_wd = train_mlp_regularized(
    train_loader, test_loader, num_epochs=num_epochs, lr=lr, weight_decay=0.01, dropout_rate=0.0
)

print("\nTraining with Dropout...")
train_loss_do, test_loss_do = train_mlp_regularized(
    train_loader, test_loader, num_epochs=num_epochs, lr=lr, weight_decay=0.0, dropout_rate=0.5
)

# 绘制对比图
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(range(1, num_epochs + 1), train_loss_base, label='Train Loss')
plt.plot(range(1, num_epochs + 1), test_loss_base, label='Test Loss')
plt.title('No Regularization')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(range(1, num_epochs + 1), train_loss_wd, label='Train Loss')
plt.plot(range(1, num_epochs + 1), test_loss_wd, label='Test Loss')
plt.title('Weight Decay (L2)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(range(1, num_epochs + 1), train_loss_do, label='Train Loss')
plt.plot(range(1, num_epochs + 1), test_loss_do, label='Test Loss')
plt.title('Dropout')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


# 4 数值稳定性和激活函数

## 4.1 理论计算题

### 1. 梯度消失与梯度爆炸

**问题描述：** 考虑一个 d 层的深层神经网络，其梯度计算包含诸如多层矩阵连乘项 $\prod_{i=t}^{d-1} \frac{\partial h^{i+1}}{\partial h^i}$。请从矩阵乘法和激活函数导数的角度，量化分析什么情况下会导致梯度爆炸，什么情况下会导致梯度消失。

**解答：**

在深层神经网络中，根据链式法则，第 $t$ 层的梯度包含了从第 $t$ 层到第 $d$ 层的连乘项：
$\frac{\partial L}{\partial h^t} = \frac{\partial L}{\partial h^d} \prod_{i=t}^{d-1} \frac{\partial h^{i+1}}{\partial h^i}$

其中，$h^{i+1} = \sigma(W^i h^i + b^i)$，$\sigma$ 为激活函数。因此，$\frac{\partial h^{i+1}}{\partial h^i} = \text{diag}(\sigma'(W^i h^i + b^i)) (W^i)^T$。

连乘项变为：
$\prod_{i=t}^{d-1} \text{diag}(\sigma'(z^i)) (W^i)^T$

*   **梯度消失 (Vanishing Gradient)：**
    *   **激活函数角度：** 如果使用 Sigmoid 或 tanh 激活函数，它们的导数最大值分别为 0.25 和 1。在深层网络中，多个小于 1 的导数值连乘，会导致梯度呈指数级衰减，最终趋近于 0。
    *   **矩阵乘法角度：** 如果权重矩阵 $W^i$ 的特征值（或奇异值）普遍小于 1，那么矩阵的连乘也会导致结果呈指数级衰减。
*   **梯度爆炸 (Exploding Gradient)：**
    *   **矩阵乘法角度：** 如果权重矩阵 $W^i$ 的初始化值过大，导致其特征值（或奇异值）普遍大于 1，那么矩阵的连乘会导致结果呈指数级增长，最终导致梯度爆炸（甚至出现 NaN）。

### 2. ReLU 缓解梯度消失

**问题描述：** 为什么改用 ReLU 激活函数可以很大程度上缓解梯度消失问题？

**解答：**

ReLU (Rectified Linear Unit) 的数学表达式为 $f(x) = \max(0, x)$。

其导数为：
$f'(x) = \begin{cases} 1, & \text{if } x > 0 \\ 0, & \text{if } x \le 0 \end{cases}$

**缓解梯度消失的原因：**
当输入 $x > 0$ 时，ReLU 的导数恒为 1。这意味着在反向传播过程中，梯度在经过 ReLU 激活函数时不会衰减（乘以 1）。因此，即使在很深的网络中，只要神经元处于激活状态（输入大于 0），梯度就可以无损地向后传递，从而很大程度上缓解了梯度消失问题。相比之下，Sigmoid 的导数最大只有 0.25，每经过一层都会使梯度至少衰减为原来的 1/4。

## 4.2 编程题

模拟数值不稳定现象，并验证不同初始化策略对深层网络的影响。

**实现要求：**
1. 构建深层网络：使用 PyTorch 的高级 API (`nn.Sequential`) 构建一个 20 层的深层全连接网络，隐藏层宽度设为 256。
2. 模拟梯度消失/爆炸：全部激活函数采用 Sigmoid，权重采用普通高斯分布初始化（如 `nn.init.normal_(m.weight, mean=0, std=1)`），输入随机数据，观察并打印前几层和后几层的梯度范数（Gradient Norm），验证梯度消失。
3. 激活函数采用 ReLU，权重采用较大的初值（如 `std=10`），观察是否发生 NaN（梯度爆炸或数值溢出）。
4. 修复与验证：使用 Xavier 初始化（`nn.init.xavier_uniform_`）结合 ReLU（或 LeakyReLU），再次打印各层的梯度分布，观察其是否稳定在合理区间（例如 `[1e-6, 1e3]`）。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# 1. 构建深层网络
def create_deep_network(num_layers=20, hidden_size=256, activation=nn.Sigmoid):
    layers = []
    layers.append(nn.Linear(256, hidden_size))
    layers.append(activation())
    for _ in range(num_layers - 2):
        layers.append(nn.Linear(hidden_size, hidden_size))
        layers.append(activation())
    layers.append(nn.Linear(hidden_size, 10))
    return nn.Sequential(*layers)

# 辅助函数：打印梯度范数
def print_gradient_norms(model, title):
    print(f"--- {title} ---")
    norms = []
    for i, layer in enumerate(model):
        if isinstance(layer, nn.Linear):
            if layer.weight.grad is not None:
                norm = layer.weight.grad.norm().item()
                norms.append(norm)
                if i < 4 or i > len(model) - 5: # 打印前几层和后几层
                    print(f"Layer {i} (Linear) Gradient Norm: {norm:.6e}")
    return norms

# 2. 模拟梯度消失 (Sigmoid + Normal Init std=1)
model_vanish = create_deep_network(activation=nn.Sigmoid)
for m in model_vanish.modules():
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=1)

x = torch.randn(32, 256)
y = torch.randint(0, 10, (32,))
criterion = nn.CrossEntropyLoss()

output = model_vanish(x)
loss = criterion(output, y)
loss.backward()
norms_vanish = print_gradient_norms(model_vanish, "Gradient Vanishing (Sigmoid + std=1)")

# 3. 模拟梯度爆炸 (ReLU + Normal Init std=10)
model_explode = create_deep_network(activation=nn.ReLU)
for m in model_explode.modules():
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=10)

x = torch.randn(32, 256)
output = model_explode(x)
loss = criterion(output, y)
loss.backward()
norms_explode = print_gradient_norms(model_explode, "Gradient Exploding (ReLU + std=10)")

# 4. 修复与验证 (ReLU + Xavier Init)
model_stable = create_deep_network(activation=nn.ReLU)
for m in model_stable.modules():
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)

x = torch.randn(32, 256)
output = model_stable(x)
loss = criterion(output, y)
loss.backward()
norms_stable = print_gradient_norms(model_stable, "Stable Gradients (ReLU + Xavier)")

# 绘制梯度范数分布图
plt.figure(figsize=(10, 6))
plt.plot(norms_vanish, label='Vanishing (Sigmoid+std=1)', marker='o')
# 梯度爆炸可能包含 NaN，过滤掉以便绘图
norms_explode_clean = [n for n in norms_explode if not np.isnan(n)]
if norms_explode_clean:
    plt.plot(norms_explode_clean, label='Exploding (ReLU+std=10)', marker='x')
plt.plot(norms_stable, label='Stable (ReLU+Xavier)', marker='s')
plt.yscale('log')
plt.xlabel('Linear Layer Index (from input to output)')
plt.ylabel('Gradient Norm (Log Scale)')
plt.title('Gradient Norms across Layers')
plt.legend()
plt.grid(True)
plt.show()


# 5 泛化表现，协变量偏移和对抗性数据

## 5.1 理论计算题

**问题描述：** 请结合实际生活中的例子（如医疗、语音识别或电商），详细阐述以下两种环境非平稳性偏移的区别与联系：
1. 协变量偏移 (Covariate Shift)：表现为 $p(x) \neq q(x)$ 但 $p(y|x) = q(y|x)$。
2. 标签偏移 (Label Shift)：表现为 $p(y) \neq q(y)$ 但 $p(x|y) = q(x|y)$。

**解答：**

**1. 协变量偏移 (Covariate Shift)**

*   **定义：** 输入特征 $x$ 的边缘分布发生了变化（$p(x) \neq q(x)$），但给定特征 $x$ 时标签 $y$ 的条件分布保持不变（$p(y|x) = q(y|x)$）。
*   **实际例子（语音识别）：** 假设我们训练一个语音识别模型，训练数据（$p$）主要来自安静的录音室环境。在测试时（$q$），模型被部署在嘈杂的街道上。此时，输入的语音特征 $x$（包含背景噪音）的分布发生了显著变化（$p(x) \neq q(x)$）。但是，如果一个人在安静环境和嘈杂环境中说出相同的词（特征 $x$ 相同，假设模型能提取出纯净语音特征），其对应的文本标签 $y$ 应该是相同的（$p(y|x) = q(y|x)$）。

**2. 标签偏移 (Label Shift)**

*   **定义：** 标签 $y$ 的边缘分布发生了变化（$p(y) \neq q(y)$），但给定标签 $y$ 时特征 $x$ 的条件分布保持不变（$p(x|y) = q(x|y)$）。
*   **实际例子（医疗诊断）：** 假设我们训练一个疾病诊断模型，训练数据（$p$）收集于流感爆发季节，此时患病（$y=1$）的比例很高。在测试时（$q$），模型在夏季使用，患病比例大幅下降（$p(y) \neq q(y)$）。然而，对于一个真正患有流感的病人（给定 $y=1$），其表现出的症状特征 $x$（如发烧、咳嗽）的分布在冬季和夏季是基本相同的（$p(x|y) = q(x|y)$）。

**区别与联系：**

*   **区别：** 协变量偏移是“原因”（特征 $x$）的分布变了，但“因果关系”（$x \rightarrow y$）没变；标签偏移是“结果”（标签 $y$）的分布变了，但“果因关系”（$y \rightarrow x$）没变。
*   **联系：** 两者都是由于训练环境和测试环境不一致导致的分布偏移问题，都会导致模型在测试集上的性能下降。在实际应用中，这两种偏移可能同时发生。解决这两种偏移通常都需要使用重要性加权（Importance Weighting）等技术来修正训练样本的权重。

## 5.2 编程题

动手模拟一个协变量偏移环境，并使用权重修正改善测试集上的预测性能。

**实现要求：**
1. 人工数据集构造：训练集 P：从正态分布 $N(-1, 1)$ 中采样 1000 个特征 $x$，标签 $y = 2x + \epsilon$（$\epsilon$ 为小噪声）。
2. 测试集 Q：从正态分布 $N(2, 1)$ 中采样 500 个特征 $x$（此时发生了明显的协变量偏移）。
3. 基线模型：用一个简单的线性回归模型直接在训练集 P 上训练，并在测试集 Q 上评估，记录均方误差（MSE）。
4. 偏移校正实现：编写一个逻辑回归分类器，将训练集 P 的样本标记为类别 0，测试集 Q 的样本标记为类别 1。
   (a) 将两组数据混合训练分类器，从而预测出每个样本属于测试集的概率 $P(test|x)$。
   (b) 根据公式计算每个训练样本的权重 $w_i \propto \frac{P(test|x_i)}{P(train|x_i)}$。
5. 加权模型训练：使用这些权重重新训练线性回归模型（加权最小二乘法），并再次在测试集 Q 上评估。对比校正前后的测试 MSE，验证校正效果。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error

# 1. 人工数据集构造
np.random.seed(42)
n_train = 1000
n_test = 500

# 训练集 P: x ~ N(-1, 1)
X_train = np.random.normal(-1, 1, (n_train, 1))
noise_train = np.random.normal(0, 0.5, (n_train, 1))
# 假设真实的非线性关系，线性模型会欠拟合，从而体现出权重的作用
# 如果是纯线性关系，权重修正效果不明显
y_train = 2 * X_train + 0.5 * X_train**2 + noise_train 

# 2. 测试集 Q: x ~ N(2, 1) (协变量偏移)
X_test = np.random.normal(2, 1, (n_test, 1))
noise_test = np.random.normal(0, 0.5, (n_test, 1))
y_test = 2 * X_test + 0.5 * X_test**2 + noise_test

# 3. 基线模型：直接在训练集上训练
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
print(f"Baseline MSE on Test Set: {mse_baseline:.4f}")

# 4. 偏移校正实现
# 构造分类数据集
X_combined = np.vstack((X_train, X_test))
# 训练集标记为 0，测试集标记为 1
y_domain = np.hstack((np.zeros(n_train), np.ones(n_test)))

# 训练逻辑回归分类器
domain_classifier = LogisticRegression()
domain_classifier.fit(X_combined, y_domain)

# 预测训练集样本属于测试集的概率 P(test|x)
# predict_proba 返回 [P(class=0), P(class=1)]
probs = domain_classifier.predict_proba(X_train)
p_train_given_x = probs[:, 0]
p_test_given_x = probs[:, 1]

# 计算权重 w_i = P(test|x_i) / P(train|x_i)
# 添加小常数防止除以 0
weights = p_test_given_x / (p_train_given_x + 1e-5)

# 归一化权重 (可选，但有助于稳定训练)
weights = weights / np.mean(weights)

# 5. 加权模型训练
weighted_model = LinearRegression()
# 使用 sample_weight 参数进行加权训练
weighted_model.fit(X_train, y_train, sample_weight=weights)
y_pred_weighted = weighted_model.predict(X_test)
mse_weighted = mean_squared_error(y_test, y_pred_weighted)
print(f"Weighted MSE on Test Set: {mse_weighted:.4f}")

# 可视化结果
plt.figure(figsize=(12, 6))

# 绘制数据分布
plt.scatter(X_train, y_train, alpha=0.3, label='Train Data (P)', color='blue', s=10)
plt.scatter(X_test, y_test, alpha=0.3, label='Test Data (Q)', color='red', s=10)

# 绘制模型预测曲线
x_range = np.linspace(-4, 5, 100).reshape(-1, 1)
plt.plot(x_range, baseline_model.predict(x_range), label='Baseline Model', color='green', linestyle='--')
plt.plot(x_range, weighted_model.predict(x_range), label='Weighted Model', color='purple', linewidth=2)

plt.xlabel('Feature x')
plt.ylabel('Label y')
plt.title('Covariate Shift Correction')
plt.legend()
plt.grid(True)
plt.show()
